In [111]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_extraction import DictVectorizer

In [112]:
dataset=r"C:\Users\Mahakaal\Documents\ChurnSense-AI\data\processed\Telco-Customer-Churn cleaned dataset.csv"
df=pd.read_csv(dataset)
df.head().T

,0,1,2,3,4
Unnamed: 0,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,1,0,0,0,0
dependents,0,0,0,0,0
tenure,1,34,2,45,2
phoneservice,0,1,1,0,1
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic


In [113]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        7043 non-null   int64  
 1   customerid        7043 non-null   object 
 2   gender            7043 non-null   object 
 3   seniorcitizen     7043 non-null   int64  
 4   partner           7043 non-null   int64  
 5   dependents        7043 non-null   int64  
 6   tenure            7043 non-null   int64  
 7   phoneservice      7043 non-null   int64  
 8   multiplelines     7043 non-null   object 
 9   internetservice   7043 non-null   object 
 10  onlinesecurity    7043 non-null   object 
 11  onlinebackup      7043 non-null   object 
 12  deviceprotection  7043 non-null   object 
 13  techsupport       7043 non-null   object 
 14  streamingtv       7043 non-null   object 
 15  streamingmovies   7043 non-null   object 
 16  contract          7043 non-null   object 


In [114]:
X = df.drop(columns=['churn'])
y = df['churn']

categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numerical_cols = X.select_dtypes(include=['number']).columns.tolist()

X_encoded = X.copy()
if categorical_cols:
    encoder = OrdinalEncoder()
    X_encoded[categorical_cols] = encoder.fit_transform(X[categorical_cols])

discrete_mask = [col in categorical_cols for col in X_encoded.columns]

mi_scores = mutual_info_classif(
    X_encoded,
    y,
    discrete_features=discrete_mask,
    random_state=42
)

mi_results = pd.Series(mi_scores, index=X_encoded.columns).sort_values(ascending=False)
print(mi_results)

customerid          0.578599
contract            0.098453
tenure              0.074846
onlinesecurity      0.064677
techsupport         0.063021
internetservice     0.055574
monthlycharges      0.047871
onlinebackup        0.046792
paymentmethod       0.044519
deviceprotection    0.043917
totalcharges        0.043876
streamingmovies     0.032001
streamingtv         0.031908
paperlessbilling    0.020218
dependents          0.019535
partner             0.016093
seniorcitizen       0.007075
phoneservice        0.004747
multiplelines       0.000801
gender              0.000037
Unnamed: 0          0.000000
dtype: float64


In [115]:
df=df.drop(columns=['gender','multiplelines','customerid','Unnamed: 0'])

In [116]:
dicts = df[df.columns[df.dtypes==object]].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
dv.fit(dicts)
y=dv.get_feature_names_out()
categorical_df=dv.transform(dicts)
categorical_df=pd.DataFrame(categorical_df, columns=y)

numerical_df=df.select_dtypes(include=[np.number])

df=pd.concat([numerical_df, categorical_df], axis=1)

In [117]:
df.head().T

,0,1,2,3,4
seniorcitizen,0.00,0.00,0.00,0.00,0.00
partner,1.00,0.00,0.00,0.00,0.00
dependents,0.00,0.00,0.00,0.00,0.00
tenure,1.00,34.00,2.00,45.00,2.00
phoneservice,0.00,1.00,1.00,0.00,1.00
paperlessbilling,1.00,0.00,1.00,0.00,1.00
monthlycharges,29.85,56.95,53.85,42.30,70.70
totalcharges,29.85,1889.50,108.15,1840.75,151.65
churn,0.00,0.00,1.00,0.00,1.00
contract=month-to-month,1.00,0.00,1.00,0.00,1.00


In [118]:
df.to_csv(r"C:\Users\Mahakaal\Documents\ChurnSense-AI\data\processed\Telco-Customer-Churn eda_completed_dataset.csv", index=False)